# 03 几何变换实战

## 概述

几何变换是图像处理中最基础也是最重要的操作之一。通过几何变换，我们可以对图像进行平移、旋转、缩放、剪切等操作，以及更高级的透视变换来矫正拍摄角度。本模块将带你从最底层的手动矩阵实现开始，逐步深入到使用 skimage 和 OpenCV 的高级变换工具，最终完成一个文档扫描的真实案例。

学习目标：
1. 理解仿射变换的矩阵表示和参数含义
2. 手动实现旋转矩阵并应用到图像
3. 使用 skimage.transform.AffineTransform 进行各种仿射变换
4. 掌握透视变换的原理和应用
5. 构建高斯金字塔和拉普拉斯金字塔
6. 实现完整的文档扫描矫正流程

In [ ]:
# ===== 环境准备 =====
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from PIL import Image
import cv2
from skimage import transform, data, color, io
from skimage.transform import (rescale, rotate, resize,
                                AffineTransform, warp, SimilarityTransform)
from ipywidgets import interact, FloatSlider, IntSlider

# 导入项目工具模块
from utils.image_utils import (load_image, display_image, display_images,
                                compare_images, ensure_uint8, ensure_float)
from utils.sample_data import checkerboard
from utils.visualization import plot_histogram

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('环境准备完毕！')
print(f'numpy 版本: {np.__version__}')
print(f'OpenCV 版本: {cv2.__version__}')

## 手动实现旋转矩阵

仿射变换可以用一个 2x3 的变换矩阵来表示。最常见的旋转矩阵形式为：

```
[ cos(theta)  -sin(theta)  tx ]
[ sin(theta)   cos(theta)  ty ]
```

其中 theta 是旋转角度（弧度），tx、ty 是平移量。当我们绕图像中心旋转时，需要先将中心平移到原点，旋转后再平移回去。下面我们从零实现这一过程。

In [ ]:
# ===== 手动实现旋转矩阵 =====

def build_rotation_matrix(angle_deg, center=(0, 0)):
    """构建绕任意中心旋转的变换矩阵"""
    theta = np.radians(angle_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    cx, cy = center
    # 先平移到原点，旋转，再平移回去
    M = np.array([
        [cos_t, -sin_t, cx - cx * cos_t + cy * sin_t],
        [sin_t,  cos_t, cy - cx * sin_t - cy * cos_t]
    ])
    return M


def manual_rotate_image(image, angle_deg):
    """使用手动旋转矩阵旋转图像（最近邻插值）"""
    h, w = image.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = build_rotation_matrix(angle_deg, center)

    # 构建完整的 3x3 矩阵并求逆（反向映射）
    M_full = np.vstack([M, [0, 0, 1]])
    M_inv = np.linalg.inv(M_full)

    # 计算旋转后的图像尺寸（保留全部内容）
    corners = np.array([[0, 0, 1], [w-1, 0, 1], [w-1, h-1, 1], [0, h-1, 1]]).T
    transformed_corners = M_full @ corners
    min_x, min_y = transformed_corners[0].min(), transformed_corners[1].min()
    max_x, max_y = transformed_corners[0].max(), transformed_corners[1].max()
    new_w, new_h = int(np.ceil(max_x - min_x)), int(np.ceil(max_y - min_y))

    result = np.zeros((new_h, new_w) + image.shape[2:], dtype=image.dtype)

    # 遍历输出像素，反向映射
    for out_y in range(new_h):
        for out_x in range(new_w):
            src_pt = M_inv @ np.array([out_x + min_x, out_y + min_y, 1])
            src_x, src_y = int(round(src_pt[0])), int(round(src_pt[1]))
            if 0 <= src_x < w and 0 <= src_y < h:
                result[out_y, out_x] = image[src_y, src_x]

    return result


# 生成棋盘格测试图并演示旋转
board = checkerboard(size=200, square_size=25)
rotated_30 = manual_rotate_image(board, 30)
rotated_60 = manual_rotate_image(board, 60)
rotated_90 = manual_rotate_image(board, 90)

display_images(
    [board, rotated_30, rotated_60, rotated_90],
    titles=['原始', '旋转 30°', '旋转 60°', '旋转 90°'],
    cols=4, figsize=(6, 4)
)
plt.suptitle('手动旋转矩阵实现效果', fontsize=14)
plt.show()

## skimage 中的 AffineTransform

skimage.transform 模块提供了 AffineTransform 类，可以方便地组合平移、旋转、缩放和剪切操作。每个变换的矩阵形式如下：

- **平移 (Translation)**：仅改变 tx, ty
- **旋转 (Rotation)**：通过 rotation 参数指定弧度
- **缩放 (Scale)**：通过 scale=(sx, sy) 指定各方向缩放因子
- **剪切 (Shear)**：通过 shear 参数指定剪切角度

组合变换时，最终矩阵为各个变换矩阵依次相乘的结果。下面我们演示所有四种基本仿射变换。

In [ ]:
# ===== skimage.transform.AffineTransform 基础演示 =====

# 加载测试图片
img = data.astronaut()
img = ensure_float(img)
h, w = img.shape[:2]
print(f'原始图片尺寸: {img.shape}')

# --- 平移 ---
tf_translate = AffineTransform(translation=(50, -30))
img_translated = warp(img, tf_translate.inverse)

# --- 旋转 ---
tf_rotate = AffineTransform(rotation=np.radians(45))
img_rotated = warp(img, tf_rotate.inverse)

# --- 缩放 ---
tf_scale = AffineTransform(scale=(0.6, 1.4))
img_scaled = warp(img, tf_scale.inverse)

# --- 剪切 ---
tf_shear = AffineTransform(shear=np.radians(25))
img_sheared = warp(img, tf_shear.inverse)

print('平移矩阵:\n', tf_translate.params)
print('\n旋转矩阵:\n', tf_rotate.params)
print('\n缩放矩阵:\n', tf_scale.params)
print('\n剪切矩阵:\n', tf_shear.params)

In [ ]:
# ===== 2x2 子图展示四种仿射变换 =====

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0, 0].imshow(ensure_uint8(img_translated))
axes[0, 0].set_title('平移 (tx=50, ty=-30)', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(ensure_uint8(img_rotated))
axes[0, 1].set_title('旋转 45°', fontsize=12)
axes[0, 1].axis('off')

axes[1, 0].imshow(ensure_uint8(img_scaled))
axes[1, 0].set_title('缩放 (sx=0.6, sy=1.4)', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(ensure_uint8(img_sheared))
axes[1, 1].set_title('剪切 25°', fontsize=12)
axes[1, 1].axis('off')

plt.suptitle('四种基本仿射变换', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 透视变换

透视变换（Perspective Transformation）比仿射变换更强大，它可以模拟"近大远小"的透视效果。透视变换使用 3x3 的单应矩阵（Homography Matrix），拥有 8 个自由度（9 个元素减去尺度不变性）。

### 与仿射变换的对比

| 特性 | 仿射变换 (2x3) | 透视变换 (3x3) |
|------|--------------|--------------|
| 保持平行线 | 是 | 否（会聚于消失点） |
| 矩形映射 | 平行四边形 | 任意四边形 |
| 最少对应点 | 3对 | 4对 |
| 典型应用 | 旋转、缩放、平移 | 文档矫正、3D投影 |

透视变换的核心公式：给定 4 对源点和目标点，计算一个 3x3 矩阵 H，使得每个源点 (x,y) 映射到目标点 (x',y')。

In [ ]:
# ===== 透视变换基础演示 =====

# 使用棋盘格来演示透视效果
board = checkerboard(size=300, square_size=30)
h, w = board.shape[:2]

# 定义四个源点（原始矩形四角）
src_pts = np.float32([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]])

# 定义四个目标点（变换后的四边形 -- 梯形效果）
margin = 40
dst_pts = np.float32([
    [margin, margin],              # 左上
    [w - 1 - margin, margin + 10], # 右上
    [w - 1 - 10, h - 1 - margin],  # 右下
    [10, h - 1 - margin]           # 左下
])

# 计算透视变换矩阵
M_persp = cv2.getPerspectiveTransform(src_pts, dst_pts)
print('透视变换矩阵 (3x3):')
print(np.round(M_persp, 4))

# 应用透视变换
board_warped = cv2.warpPerspective(board, M_persp, (w, h))

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].imshow(board)
axes[0].set_title('原始棋盘格 + 目标四边形')
poly = Polygon(dst_pts, fill=False, edgecolor='red', linewidth=2)
axes[0].add_patch(poly)
for i, pt in enumerate(dst_pts):
    axes[0].plot(pt[0], pt[1], 'ro', markersize=6)
    axes[0].text(pt[0] + 8, pt[1] + 8, f'P{i+1}', color='red', fontsize=11)
axes[0].axis('off')

axes[1].imshow(board_warped)
axes[1].set_title('透视变换结果')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 文档扫描模拟

这是本模块最核心的实战案例 —— 模拟"扫描全能王"的核心功能。整体流程如下：

1. **模拟拍照**：生成一个模拟的文档图片，并施加透视扭曲
2. **边缘检测**：使用 Canny 算法检测文档边缘
3. **角点定位**：查找最大轮廓并用多边形近似得到四个角点
4. **透视矫正**：用 4 对角点计算透视变换矩阵并矫正
5. **增强处理**：对比度增强，模拟扫描效果

In [ ]:
# ===== 创建模拟文档图片 =====

def create_document_image(text_lines=12, doc_size=(600, 450)):
    """生成一张模拟的文档图片"""
    img = Image.new('RGB', doc_size, color=(255, 255, 255))
    from PIL import ImageDraw
    draw = ImageDraw.Draw(img)

    # 标题
    draw.text((50, 25), '年度工作报告', fill=(0, 0, 0))
    draw.line([(50, 55), (doc_size[0] - 50, 55)], fill=(0, 0, 0), width=2)

    # 正文行
    for i in range(text_lines):
        y = 80 + i * 28
        line_w = np.random.randint(doc_size[0] * 3 // 5, doc_size[0] - 100)
        draw.rectangle([(50, y), (line_w, y + 3)], fill=(40, 40, 40))

    return np.array(img)


doc_original = create_document_image(text_lines=12)
h, w = doc_original.shape[:2]

# 模拟从右上方倾斜拍摄
src_corners = [[35, 25], [w - 45, 55], [w - 15, h - 65], [65, h - 35]]
dst_corners = np.float32([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]])
M_tilt = cv2.getPerspectiveTransform(np.float32(src_corners), dst_corners)
doc_tilted = cv2.warpPerspective(doc_original, M_tilt, (w, h), borderValue=(210, 210, 210))

compare_images(doc_original, doc_tilted,
               title_orig='原始平整文档', title_processed='模拟倾斜拍照')
plt.show()

In [ ]:
# ===== 文档扫描矫正完整流程 =====

def scan_document(image):
    """完整的文档扫描管线"""
    steps = {}
    steps['input'] = image.copy()

    # 1. 灰度化
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    steps['gray'] = gray

    # 2. 高斯模糊
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    steps['blur'] = blur

    # 3. Canny 边缘检测
    edges = cv2.Canny(blur, 40, 120)
    steps['edges'] = edges

    # 4. 膨胀使边缘闭合
    kernel = np.ones((3, 3), np.uint8)
    edges_d = cv2.dilate(edges, kernel, iterations=1)
    steps['dilated'] = edges_d

    # 5. 查找最大轮廓
    contours, _ = cv2.findContours(edges_d, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image, steps
    largest = max(contours, key=cv2.contourArea)

    # 6. 多边形近似
    peri = cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, 0.02 * peri, True)

    if len(approx) != 4:
        rect = cv2.minAreaRect(largest)
        approx = cv2.boxPoints(rect)
    else:
        approx = approx.reshape(4, 2)
    steps['corners'] = approx

    # 7. 角点排序：左上、右上、右下、左下
    pts = approx.astype(np.float32)
    s = pts.sum(axis=1)
    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = pts[np.argmin(s)]
    ordered[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    ordered[1] = pts[np.argmin(diff)]
    ordered[3] = pts[np.argmax(diff)]

    (tl, tr, br, bl) = ordered
    w_out = max(int(np.linalg.norm(tr - tl)), int(np.linalg.norm(br - bl)))
    h_out = max(int(np.linalg.norm(bl - tl)), int(np.linalg.norm(br - tr)))

    dst_pts = np.float32([[0, 0], [w_out-1, 0], [w_out-1, h_out-1], [0, h_out-1]])
    M = cv2.getPerspectiveTransform(ordered, dst_pts)
    corrected = cv2.warpPerspective(image, M, (w_out, h_out))
    steps['corrected'] = corrected

    # 8. CLAHE 增强
    gray_c = cv2.cvtColor(corrected, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray_c)
    steps['enhanced'] = enhanced

    return corrected, enhanced, steps


corrected, enhanced, steps = scan_document(doc_tilted)

# 可视化管线各阶段
stage_imgs = [steps['input'], steps['edges'], steps['corrected'], enhanced]
stage_names = ['输入（倾斜文档）', 'Canny 边缘检测', '透视矫正结果', 'CLAHE 对比度增强']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, img, title in zip(axes.flat, stage_imgs, stage_names):
    if len(img.shape) == 2:
        ax.imshow(img, cmap='gray')
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis('off')
plt.suptitle('文档扫描管线各阶段', fontsize=14)
plt.tight_layout()
plt.show()

print(f'检测到的角点坐标: \n{steps["corners"]}')

## 图像金字塔

图像金字塔是同一图像在不同分辨率下的表示集合。主要有两种：

### 高斯金字塔（Gaussian Pyramid）
通过反复进行高斯平滑 + 下采样构建。每一层是上一层的 1/4 大小。第 i+1 层的像素是通过对第 i 层进行高斯模糊后隔行隔列采样得到的。

### 拉普拉斯金字塔（Laplacian Pyramid）
拉普拉斯金字塔的每一层 = 高斯金字塔第 i 层 - 第 i+1 层上采样回原尺寸。它保存的是"残差"——被下采样丢弃的高频细节信息。

数学表达：L_i = G_i - Up(G_{i+1})

应用场景：多尺度特征提取（SIFT）、图像融合（Multi-band Blending）、图像压缩。

In [ ]:
# ===== 构建高斯金字塔 =====

def build_gaussian_pyramid(image, levels=4):
    """构建高斯金字塔"""
    pyramid = [image.copy()]
    current = image.copy()
    for i in range(levels - 1):
        current = cv2.pyrDown(current)
        pyramid.append(current)
    return pyramid


# 使用 skimage 内置图片
img_cam = data.camera()
print(f'原始图片尺寸: {img_cam.shape}')

g_pyr = build_gaussian_pyramid(img_cam, levels=4)

# 在一行中显示各层
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, g) in enumerate(zip(axes, g_pyr)):
    ax.imshow(g, cmap='gray')
    ax.set_title(f'高斯层 {i}\n({g.shape[0]} x {g.shape[1]})', fontsize=11)
    ax.axis('off')
plt.suptitle('高斯金字塔（逐层下采样）', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 构建拉普拉斯金字塔 =====

def build_laplacian_pyramid(image, levels=4):
    """构建拉普拉斯金字塔"""
    g_pyr = build_gaussian_pyramid(image, levels)
    lap_pyr = []
    for i in range(levels - 1):
        # 将 i+1 层上采样到与 i 层相同尺寸
        up = cv2.pyrUp(g_pyr[i + 1], dstsize=(g_pyr[i].shape[1], g_pyr[i].shape[0]))
        # 残差
        lap = cv2.subtract(g_pyr[i].astype(np.int16), up.astype(np.int16))
        lap_pyr.append(lap)
    lap_pyr.append(g_pyr[-1])  # 最小层直接保留
    return lap_pyr, g_pyr


lap_pyr, g_pyr = build_laplacian_pyramid(img_cam, levels=4)

# 可视化拉普拉斯金字塔
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, lap) in enumerate(zip(axes, lap_pyr)):
    if i < 3:
        vmax = max(abs(lap.min()), abs(lap.max()))
        im = ax.imshow(lap, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.85)
        ax.set_title(f'拉普拉斯层 {i} (残差)', fontsize=11)
    else:
        ax.imshow(lap, cmap='gray')
        ax.set_title(f'拉普拉斯层 {i} (底层)', fontsize=11)
    ax.axis('off')
plt.suptitle('拉普拉斯金字塔（残差 = 高频细节）', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 交互式仿射变换参数探索 =====

img_demo = data.astronaut()
img_demo_f = ensure_float(img_demo)

def explore_affine(rotation=0.0, scale_x=1.0, scale_y=1.0,
                   shear_deg=0.0, translate_x=0, translate_y=0):
    """交互式探索仿射变换参数"""
    tf = AffineTransform(
        rotation=np.radians(rotation),
        scale=(scale_x, scale_y),
        shear=np.radians(shear_deg),
        translation=(translate_x, translate_y)
    )
    warped = warp(img_demo_f, tf.inverse)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(ensure_uint8(img_demo_f))
    ax1.set_title('原始图片')
    ax1.axis('off')
    ax2.imshow(ensure_uint8(warped))
    ax2.set_title(f'变换后\n旋转={rotation} deg, 缩放=({scale_x},{scale_y}), 剪切={shear_deg} deg')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'变换矩阵 (3x3):\n{tf.params}')

interact(
    explore_affine,
    rotation=FloatSlider(value=0, min=-180, max=180, step=5, description='旋转(度)'),
    scale_x=FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1, description='X缩放'),
    scale_y=FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1, description='Y缩放'),
    shear_deg=FloatSlider(value=0, min=-45, max=45, step=1, description='剪切(度)'),
    translate_x=IntSlider(value=0, min=-200, max=200, step=10, description='X平移'),
    translate_y=IntSlider(value=0, min=-200, max=200, step=10, description='Y平移'),
)


## 练习与实践

完成以下练习来巩固对几何变换的理解。

In [ ]:
# 练习1: 实现图像的任意角度旋转，并观察不同插值方法的效果差异
# TODO: 完成以下练习

# 旋转角度列表
angles = [0, 30, 60, 90, 135, 180]
# 插值方法：order=0(最近邻), order=1(双线性), order=3(双三次)
interpolation_methods = [0, 1, 3]
interpolation_names = ['最近邻 (order=0)', '双线性 (order=1)', '双三次 (order=3)']

img_demo = data.camera()[:200, :200]

fig, axes = plt.subplots(len(interpolation_methods), len(angles),
                         figsize=(14, 8))

for row, (order, name) in enumerate(zip(interpolation_methods, interpolation_names)):
    for col, angle in enumerate(angles):
        # ============================================
        # TODO: 使用 skimage.transform.rotate 旋转图像
        # 参数: image, angle, resize=True, order=order, mode='constant'
        # ============================================
        rotated = rotate(img_demo, angle, resize=True, order=order, mode='constant')

        axes[row, col].imshow(rotated, cmap='gray')
        if row == 0:
            axes[row, col].set_title(f'{angle}°', fontsize=10)
        if col == 0:
            axes[row, col].set_ylabel(name, fontsize=10)
        axes[row, col].axis('off')

plt.suptitle('不同旋转角度和插值方法的效果对比', fontsize=14)
plt.tight_layout()
plt.show()

print('观察要点：')
print('- 最近邻插值在大角度旋转时会出现明显的锯齿')
print('- 双三次插值最平滑，但计算量最大')
print('- 双线性插值是常用的折中方案')

In [ ]:
# 练习2: 尝试构建一个5层的图像金字塔，并验证向上采样重建后的图像质量
# TODO: 完成以下练习

# 使用一张图片构建 5 层高斯金字塔，然后从最底层逐层上采样重建
img_test = data.camera().astype(np.float32)
h, w = img_test.shape

# 构建 5 层高斯金字塔
NUM_LEVELS = 5
g_pyr_5 = [img_test.copy()]
current = img_test.copy()
for i in range(NUM_LEVELS - 1):
    current = cv2.pyrDown(current)
    g_pyr_5.append(current)

print('各层尺寸:')
for i, g in enumerate(g_pyr_5):
    print(f'  层 {i}: {g.shape}')

# TODO: 从最底层逐层上采样，尝试重建原始图像
# 逐层 pyrUp，到达与原图相同尺寸
reconstructed = g_pyr_5[-1].copy()
for i in range(NUM_LEVELS - 1):
    target_shape = g_pyr_5[NUM_LEVELS - 2 - i].shape
    reconstructed = cv2.pyrUp(reconstructed, dstsize=(target_shape[1], target_shape[0]))

# 计算重建误差
diff = np.abs(img_test - reconstructed)
print(f'\n重建误差统计:')
print(f'  最大误差: {diff.max():.2f}')
print(f'  平均误差: {diff.mean():.2f}')

# 可视化原始、重建和误差
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_test, cmap='gray')
axes[0].set_title('原始图像')
axes[0].axis('off')

axes[1].imshow(reconstructed, cmap='gray')
axes[1].set_title(f'从 5 层金字塔重建\n(尺寸: {reconstructed.shape[0]}x{reconstructed.shape[1]})')
axes[1].axis('off')

axes[2].imshow(diff, cmap='hot')
axes[2].set_title(f'误差图 (max={diff.max():.1f})')
axes[2].axis('off')

plt.suptitle('金字塔重建质量验证', fontsize=14)
plt.tight_layout()
plt.show()

print('\n观察要点：')
print('- 上采样重建会丢失高频细节（模糊）')
print('- 拉普拉斯金字塔保存的正是这些丢失的细节')
print('- 层数越多，重建质量越差')

## 实战案例：文档扫描校正应用

现在我们综合运用所学的所有几何变换技术，构建一个完整的文档扫描类。该类的核心功能：

1. 接收倾斜拍摄的文档照片
2. 自动检测文档的四个角点
3. 计算透视变换矩阵并矫正
4. 增强对比度模拟扫描效果
5. 可选二值化输出

这是一个实际可用的文档扫描方案，涵盖了本模块的所有核心技术。

In [ ]:
# ===== 完整的文档扫描类 =====

class DocumentScanner:
    """一站式文档扫描器"""

    def __init__(self):
        self.steps = {}

    def scan(self, image, enhance=True, binarize=False):
        """执行完整扫描流程"""
        self.steps['input'] = image.copy()
        h, w = image.shape[:2]

        # 1. 灰度化 + 高斯模糊
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        self.steps['gray'] = blur

        # 2. Canny 边缘检测
        edges = cv2.Canny(blur, 50, 150)
        self.steps['edges'] = edges

        # 3. 闭运算闭合缺口
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
        self.steps['closed'] = closed

        # 4. 查找轮廓
        contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print('警告：未找到轮廓')
            return image

        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]

        # 5. 寻找四边形
        doc_corners = None
        for cnt in contours:
            peri = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
            if len(approx) == 4:
                doc_corners = approx.reshape(4, 2)
                break
        if doc_corners is None:
            rect = cv2.minAreaRect(contours[0])
            doc_corners = cv2.boxPoints(rect)
        self.steps['corners'] = doc_corners.astype(np.float32)

        # 6. 角点排序
        pts = self.steps['corners']
        s = pts.sum(axis=1)
        ordered = np.zeros((4, 2), dtype=np.float32)
        ordered[0] = pts[np.argmin(s)]
        ordered[2] = pts[np.argmax(s)]
        diff = np.diff(pts, axis=1)
        ordered[1] = pts[np.argmin(diff)]
        ordered[3] = pts[np.argmax(diff)]

        # 7. 计算目标尺寸
        (tl, tr, br, bl) = ordered
        w_out = max(int(np.linalg.norm(tr - tl)), int(np.linalg.norm(br - bl)))
        h_out = max(int(np.linalg.norm(bl - tl)), int(np.linalg.norm(br - tr)))

        # 8. 透视矫正
        dst_pts = np.float32([[0, 0], [w_out-1, 0], [w_out-1, h_out-1], [0, h_out-1]])
        M = cv2.getPerspectiveTransform(ordered, dst_pts)
        corrected = cv2.warpPerspective(image, M, (w_out, h_out))
        self.steps['corrected'] = corrected

        result = corrected

        # 9. CLAHE 增强
        if enhance:
            gray_c = cv2.cvtColor(corrected, cv2.COLOR_RGB2GRAY)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            enhanced = clahe.apply(gray_c)
            self.steps['enhanced'] = enhanced
            result = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)

        # 10. 二值化
        if binarize:
            gray_r = cv2.cvtColor(result, cv2.COLOR_RGB2GRAY)
            _, binary = cv2.threshold(gray_r, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            self.steps['binary'] = binary
            result = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)

        return result

    def show_pipeline(self):
        """可视化扫描管线的每个步骤"""
        names = ['input', 'gray', 'edges', 'closed', 'corrected']
        if 'enhanced' in self.steps:
            names.append('enhanced')
        if 'binary' in self.steps:
            names.append('binary')

        n = len(names)
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
        if n == 1:
            axes = [axes]
        for ax, name in zip(axes, names):
            img = self.steps[name]
            if len(img.shape) == 2:
                ax.imshow(img, cmap='gray')
            else:
                ax.imshow(img)
            ax.set_title(name, fontsize=11)
            ax.axis('off')
        plt.suptitle('文档扫描管线', fontsize=14)
        plt.tight_layout()
        plt.show()


# 测试扫描器
scanner = DocumentScanner()
result = scanner.scan(doc_tilted, enhance=True, binarize=True)
scanner.show_pipeline()

print('文档扫描完成！')
print(f'输入尺寸: {doc_tilted.shape}')
print(f'输出尺寸: {result.shape}')
print('\n管线各步骤: input -> gray -> edges -> closed -> corrected -> enhanced -> binary')

In [ ]:
# ===== 多角度扫描测试 =====

# 模拟不同拍照角度
angle_configs = [
    ('右上倾斜', [[40, 20], [w-50, 60], [w-30, h-40], [70, h-20]]),
    ('左侧倾斜', [[10, 40], [w-20, 10], [w-40, h-10], [30, h-30]]),
    ('仰角拍摄', [[80, 30], [w-80, 30], [w-20, h-10], [20, h-10]]),
]

fig, axes = plt.subplots(len(angle_configs), 3, figsize=(14, 4 * len(angle_configs)))

for row, (name, corners) in enumerate(angle_configs):
    # 生成倾斜图片
    dst_c = np.float32([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]])
    M_t = cv2.getPerspectiveTransform(np.float32(corners), dst_c)
    tilted = cv2.warpPerspective(doc_original, M_t, (w, h), borderValue=(210, 210, 210))

    # 扫描矫正
    s = DocumentScanner()
    corrected = s.scan(tilted, enhance=False)

    axes[row, 0].imshow(doc_original)
    axes[row, 0].set_title(f'{name}\n原始文档', fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(tilted)
    axes[row, 1].set_title('模拟拍照', fontsize=11)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(corrected)
    axes[row, 2].set_title('矫正结果', fontsize=11)
    axes[row, 2].axis('off')

plt.suptitle('多角度文档扫描测试', fontsize=16)
plt.tight_layout()
plt.show()

## 总结与延伸

### 本模块核心要点

| 知识点 | 核心概念 | 关键函数/类 |
|--------|---------|-----------|
| 仿射变换 | 2x3 矩阵，6 自由度 | AffineTransform, warp |
| 透视变换 | 3x3 单应矩阵，8 自由度 | getPerspectiveTransform, warpPerspective |
| 图像矫正 | 四角点检测 + 透视变换 | findContours, approxPolyDP |
| 高斯金字塔 | 反复下采样 | cv2.pyrDown |
| 拉普拉斯金字塔 | 残差图像 | cv2.pyrUp + 相减 |

### 延伸阅读

- OpenCV 官方教程：Geometric Transformations of Images
- 《Multiple View Geometry in Computer Vision》-- Hartley & Zisserman
- 图像拼接/全景图（Panorama）技术
- AR（增强现实）中的 Marker 检测与姿态估计
- OCR 预处理中的文档矫正技术

### 下一模块预告

模块 04 将进入图像分割与形态学的世界：Otsu 大津法、自适应阈值、连通域分析、分水岭算法、形态学操作、工业零件计数实战。